## 문항 1 ) 할리스커피 매장 정보 10페이지 수집

In [ ]:
# 대상: https://www.hollys.co.kr/store/korea/korStore2.do

# 총 10페이지를 순회할 것 (페이지를 넘기며 URL이 바뀌는 것을 확인)

# 추출 필드: 지역 / 매장명 / 현황 / 주소 / 매장 서비스 / 전화번호

# 매장 서비스는 리스트로 담을 것 (아이콘이 여러 개인 매장이 있음)

# 결과를 hollys.csv로 저장할 것

In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import time

In [14]:
URL = 'https://www.hollys.co.kr/store/korea/korStore2.do'
PARAMS = {
    'sindo': '',
    'gugun': '',
    'store': '',  
}

def get_text(tag):
    return tag.text.strip() if tag else ''


def fetch(page):
    res = requests.get(URL, params={**PARAMS, 'pageNo': page})
    res.raise_for_status()
    return res.text

def parse(html):
    soup = BeautifulSoup(html, 'html.parser')
    trs = soup.select('.tb_store tbody tr')
    store_infos = []
    for tr in trs:
        tds = tr.select('td')
        imgs = tds[4].select('img')
        service = [img.attrs['alt'] for img in imgs]
        store_infos.append({
            '지역': get_text(tds[0]),
            '매장명': get_text(tds[1]),
            '현황': get_text(tds[2]),
            '주소': get_text(tds[3]),
            '매장서비스': service,
            '전화번호': get_text(tds[5]),
        })
    return store_infos

result = []
for page in range(1,11):
    store_info = parse(fetch(page))
    result.extend(store_info)
    print(f'{page}페이지, {len(result)}건 수집')

    time.sleep(0.7)

pd.DataFrame(result).to_csv('hollys.csv', index=False, encoding='utf-8-sig')


1페이지, 10건 수집
2페이지, 20건 수집
3페이지, 30건 수집
4페이지, 40건 수집
5페이지, 50건 수집
6페이지, 60건 수집
7페이지, 70건 수집
8페이지, 80건 수집
9페이지, 90건 수집
10페이지, 100건 수집


### 문항 2 ) 알라딘 베스트셀러 수집

In [ ]:
# 대상: 알라딘 베스트셀러 목록 페이지(https://www.aladin.co.kr/shop/common/wbest.aspx?BranchType=1)

# 추출 필드: 카테고리 / 제목 / 저자 / 할인가격 / 이미지 URL

# 이미지 URL은 <img> 태그의 속성에서 가져올 것

# 결과를 aladin_bestseller.csv로 저장할 것

# 추가학습: 총 500위까지 수집해주세요.

In [26]:
URL = "https://www.aladin.co.kr/shop/common/wbest.aspx"
PARAMS = {
    'BestType': 'Bestseller',
    'BranchType': 1,
    'CID': 0,
    'cnt': 1000,
    'SortOrder':1
}

def get_text(tag):
    return tag.text.strip() if tag else ''

def fetch(page):
    res = requests.get(URL, params={**PARAMS,'page':page})
    res.raise_for_status()
    return res.text
#Myform > div:nth-child(1) > ul > li:nth-child(3) > a:nth-child(1)
#Myform > div:nth-child(1) > ul > li:nth-child(2) > a:nth-child(1)
def parse(html):
    soup = BeautifulSoup(html, 'html.parser')
    infos = soup.select('div.ss_book_box')
    books = []
    for info in infos:
        img = info.select_one('.cover_area img, .cover_area_other img')
        if info.select_one('.ss_ht1'):
            author_selector = 'li:nth-child(3) > a:nth-child(1)'
        else:
            author_selector = 'li:nth-child(2) > a:nth-child(1)'
        books.append({
            '카테고리': get_text(info.select_one('span.tit_category')),
            '제목': get_text(info.select_one('a.bo3')),
            '저자': get_text(info.select_one(author_selector)),
            '할인가격': get_text(info.select_one('.ss_p2')),
            '이미지 URL': img.attrs['src'],

        })
    return books

ranking = []
for page in range(1, 11):
    result = parse(fetch(page))
    ranking.extend(result)
    print(f"누적 : {len(ranking)}건 수집")
    time.sleep(0.7)

pd.DataFrame(ranking).to_csv("aladin_bestseller.csv", index=False, encoding="utf-8-sig")


누적 : 50건 수집
누적 : 100건 수집
누적 : 150건 수집
누적 : 200건 수집
누적 : 250건 수집
누적 : 300건 수집
누적 : 350건 수집
누적 : 400건 수집
누적 : 450건 수집
누적 : 500건 수집
